---
title: "并查集"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

## 📝 题目 128：最长连续序列

::: {.callout-caution}
### 📖 题目描述
给定一个未排序的整数数组 `nums` ，找出数字连续的最长序列（不要求序列元素在原数组中连续）的长度。

**⚠️ 绝对死线警告**：
请你设计并实现时间复杂度为 **$O(n)$** 的算法解决此问题。

**输入输出示例**：

* **示例 1**：
    * **输入**：`nums = [100, 4, 200, 1, 3, 2]`
    * **输出**：`4`
    * **解释**：最长数字连续序列是 `[1, 2, 3, 4]`。它的长度为 4。

* **示例 2**：
    * **输入**：`nums = [0, 3, 7, 2, 5, 8, 4, 6, 0, 1]`
    * **输出**：`9`
    * **解释**：最长数字连续序列是 `[0, 1, 2, 3, 4, 5, 6, 7, 8]`。
:::

In [1]:
class Solution128:
    def longestConsecutive(self, nums: list[int]) -> int:
        # 1. 空间换时间：把所有数字扔进哈希表，顺便去重，查询时间降至 O(1)
        num_set = set(nums)
        longest_streak = 0

        for num in num_set:
            # 2. 只从“序列的绝对起点”开始找，如果 num - 1 在集合里，说明当前数字只是别人的小弟，直接跳过！
            if num - 1 not in num_set:
                current_num = num
                current_streak = 1

                # 3. 既然是起点，就一路往上查，看这个序列到底有多长
                while current_num + 1 in num_set:
                    current_num += 1
                    current_streak += 1

                # 4. 更新历史最高纪录
                longest_streak = max(longest_streak, current_streak)

        return longest_streak

In [7]:
#| code-fold: true
def test_128(func):
    test_cases = [
        {"desc": "1. 示例 1: 经典乱序", "nums": [100, 4, 200, 1, 3, 2], "expected": 4},
        {"desc": "2. 示例 2: 包含 0 和重复", "nums": [0, 3, 7, 2, 5, 8, 4, 6, 0, 1], "expected": 9},
        {"desc": "3. 边界: 空数组", "nums": [], "expected": 0},
        {"desc": "4. 边界: 只有一个元素", "nums": [5], "expected": 1},
        {"desc": "5. 边界: 全是重复元素", "nums": [2, 2, 2, 2, 2], "expected": 1},
        {"desc": "6. 跨越 0 和负数的序列", "nums": [2, -3, -1, 1, 0, -2], "expected": 6},
        {"desc": "7. 完全断开的极大数字", "nums": [10, 20, 30, 40], "expected": 1},
        {"desc": "8. 两段长度一样的序列", "nums": [1, 2, 3, 10, 11, 12], "expected": 3},
        {"desc": "9. 大厂压测: 倒序的连续极长序列", "nums": list(range(10000, 0, -1)), "expected": 10000}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<8} | {'实际':<8} | {'描述'}")
    print("-" * 80)

    for i, tc in enumerate(test_cases):
        actual = func(tc["nums"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<8} | {str(actual):<8} | {tc['desc']}")

    print("-" * 80)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_128(Solution128().longestConsecutive)

ID  | 结果     | 预期       | 实际       | 描述
--------------------------------------------------------------------------------
1   | ✅ PASS | 4        | 4        | 1. 示例 1: 经典乱序
2   | ✅ PASS | 9        | 9        | 2. 示例 2: 包含 0 和重复
3   | ✅ PASS | 0        | 0        | 3. 边界: 空数组
4   | ✅ PASS | 1        | 1        | 4. 边界: 只有一个元素
5   | ✅ PASS | 1        | 1        | 5. 边界: 全是重复元素
6   | ✅ PASS | 6        | 6        | 6. 跨越 0 和负数的序列
7   | ✅ PASS | 1        | 1        | 7. 完全断开的极大数字
8   | ✅ PASS | 3        | 3        | 8. 两段长度一样的序列
9   | ✅ PASS | 10000    | 10000    | 9. 大厂压测: 倒序的连续极长序列
--------------------------------------------------------------------------------
测试总结: 通过 9/9


::: {.callout-important}
### 💡 思路讲解

1. **构建上帝视角（空间换时间）**：
   - 将原始数组 `nums` 转换为哈希集合 `num_set`（如 Python 中的 `set(nums)`）。
   - **目的**：利用哈希表特性进行去重，并将“查询任意数字是否存在”的时间复杂度从 $O(n)$ 瞬间降维至极速的 $O(1)$。

2. **寻找序列起点（精准拦截逻辑）**：
   - 遍历哈希集合中的每一个数字 `num`，寻找连续序列的带头人。
   - **起点判定**：检查 `num - 1` 是否存在于集合中。如果存在，说明当前 `num` 只是跟在别人后面的“小弟”（处于序列中间或尾部），直接跳过不处理；只有当 `num - 1` 不存在时，才认定 `num` 是一条新序列的绝对起点。

3. **顺藤摸瓜（链式探索）**：
   - 当确定当前数字 `current_num = num` 为起点时，初始化当前序列长度 `current_streak = 1`。
   - **向后探索**：开启 `while` 循环，不断在集合中查询下一个连续数字 `current_num + 1` 是否存在。若存在，则数字递增，序列长度 `current_streak` 随之加 1，直至发生断链。

4. **全局结果更新**：
   - 在每一轮完整的“顺藤摸瓜”结束后，将刚刚获取到的序列长度 `current_streak` 与全局历史最长记录 `longest_streak` 进行比较并打擂台，保留最大值。最终遍历结束返回该最大值。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(n)$。其中 $n$ 为数组的长度。虽然代码中存在嵌套的 `while` 循环，但得益于“起点判定”拦截器，内层 `while` 循环仅在遇到序列起点时才会执行。这保证了集合中的每个元素最多被外层遍历访问 1 次，被内层延展访问 1 次，整体操作次数为线性，均摊时间复杂度严格等于 $O(n)$。
* **空间复杂度**：$O(n)$。为了实现 $O(1)$ 的查询速度，算法开辟了一个额外的数据结构（哈希集合）来存储所有的独立数字。在最坏情况下（即数组中没有任何重复元素），哈希表需要占据 $O(n)$ 的额外空间。
:::

## 📝 题目 560：和为 K 的子数组

::: {.callout-caution}
### 📖 题目描述
给你一个整数数组 `nums` 和一个整数 `k` ，请你统计并返回 该数组中和为 `k` 的连续子数组的个数。
**注意**：数组中可能包含**负数**和**零**，这意味着你**绝对不能使用滑动窗口**（因为右指针向右移，总和不一定增加；左指针向右移，总和也不一定减少，窗口的单调性被破坏了）。

**输入输出示例**：

* **示例 1**：
    * **输入**：`nums = [1,1,1]`, `k = 2`
    * **输出**：`2`
    * **解释**：[1,1] (前两个) 和 [1,1] (后两个) 和都为 2。

* **示例 2**：
    * **输入**：`nums = [1,2,3]`, `k = 3`
    * **输出**：`2`
    * **解释**：[1,2] 和 [3] 和都为 3。
:::

In [3]:
class Solution560:
    def subarraySum(self, nums: list[int], k: int) -> int:
        # 1. 初始化哈希账本，{0: 1} 是极其关键的“虚拟原点”
        prefix_dict = {0: 1}

        curr_sum = 0
        count = 0

        # 2. 开始遍历
        for num in nums:
            curr_sum += num  # 计算当前位置的前缀和

            # 3. 看看历史中有没有我们需要的目标前缀和
            target = curr_sum - k
            if target in prefix_dict:
                count += prefix_dict[target]

            # 4. 把自己当前的前缀和存入账本，供后面的人查询
            prefix_dict[curr_sum] = prefix_dict.get(curr_sum, 0) + 1

        return count

In [8]:
#| code-fold: true
def test_560(func):
    test_cases = [
        {"desc": "1. 基础全是正数", "nums": [1, 1, 1], "k": 2, "expected": 2},
        {"desc": "2. 包含单个直接命中", "nums": [1, 2, 3], "k": 3, "expected": 2},
        {"desc": "3. {0:1} 的威力 (从头开始正好等于k)", "nums": [3, 4, 7, -2, 2], "k": 7, "expected": 3},
        {"desc": "4. 负数抵消的魔法 (滑动窗口的死穴)", "nums": [1, -1, 1, -1, 1], "k": 1, "expected": 6},
        {"desc": "5. 满地都是 0 (考验前缀和次数累加)", "nums": [0, 0, 0, 0], "k": 0, "expected": 10},
        {"desc": "6. 目标为 0 且带负数", "nums": [1, 2, -3, 3, -2, -1], "k": 0, "expected": 5},
        {"desc": "7. 完全找不到目标", "nums": [2, 4, 6, 8], "k": 3, "expected": 0},
        {"desc": "8. 单个元素正好命中", "nums": [5], "k": 5, "expected": 1},
        {"desc": "9. 单个元素不命中", "nums": [5], "k": 2, "expected": 0},
        {"desc": "10. 大长串连续重复数字寻找区间", "nums": [2, 2, 2, 2, 2], "k": 4, "expected": 4}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 75)

    for i, tc in enumerate(test_cases):
        actual = func(tc["nums"], tc["k"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<4} | {str(actual):<4} | {tc['desc']}")

    print("-" * 75)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_560(Solution560().subarraySum)

ID  | 结果     | 预期   | 实际   | 描述
---------------------------------------------------------------------------
1   | ✅ PASS | 2    | 2    | 1. 基础全是正数
2   | ✅ PASS | 2    | 2    | 2. 包含单个直接命中
3   | ✅ PASS | 3    | 3    | 3. {0:1} 的威力 (从头开始正好等于k)
4   | ✅ PASS | 6    | 6    | 4. 负数抵消的魔法 (滑动窗口的死穴)
5   | ✅ PASS | 10   | 10   | 5. 满地都是 0 (考验前缀和次数累加)
6   | ✅ PASS | 5    | 5    | 6. 目标为 0 且带负数
7   | ✅ PASS | 0    | 0    | 7. 完全找不到目标
8   | ✅ PASS | 1    | 1    | 8. 单个元素正好命中
9   | ✅ PASS | 0    | 0    | 9. 单个元素不命中
10  | ✅ PASS | 4    | 4    | 10. 大长串连续重复数字寻找区间
---------------------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

1. **公式转换（前缀和降维）**：
   - **核心等式**：子数组 `[j, i]` 的和，等于“从头到 `i` 的前缀和”减去“从头到 `j-1` 的前缀和”。
   - 题目要求寻找和为 `k` 的子数组，即 `前缀和[i] - 前缀和[j-1] == k`。
   - **数学移项**：`前缀和[j-1] == 前缀和[i] - k`。这意味着，当我们在位置 `i` 算出当前前缀和时，不需要写内层循环回头去找起点，只需查阅历史记录：“以前出现过值为 `当前前缀和 - k` 的前缀和吗？”

2. **初始化哈希账本（虚拟原点）**：
   - 创建哈希表 `prefix_dict`，用于记录 `{"某一个前缀和的值": "它出现的次数"}`。
   - **极其关键的初始化**：必须先向哈希表中存入 `{0: 1}`。这代表“前缀和为 0 的情况出现过 1 次”（可以理解为在数组第 0 个元素之前的虚拟原点）。这是为了应对“从头开始累加的子数组和正好完全等于 `k`”的合法情况。

3. **一次遍历与动态更新**：
   - 遍历数组，维护一个变量 `curr_sum`，不断累加当前遍历到的数字。
   - **先查账（历史匹配）**：计算目标前缀和 `target = curr_sum - k`。如果 `target` 存在于哈希表中，说明找到了合法的子数组，将其对应的出现次数累加到结果集 `count` 中。
   - **后记账（登记自己）**：将当前的 `curr_sum` 登记到哈希表中。如果已存在则次数加 1，不存在则设为 1。（注意顺序，必须先查账后记账，防止 `k=0` 时自己匹配自己）。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：O(N)。其中 N 为数组的长度。我们只需遍历数组一次，并且 Python 字典（哈希表）的查询和插入操作平均时间复杂度都是极速的 O(1)。
* **空间复杂度**：O(N)。在最坏情况下（例如数组元素全是正数，导致每个前缀和都不一样），哈希表中需要同时存储 O(N) 个不同的前缀和键值对。
:::

## 📝 题目 3：无重复字符的最长子串

::: {.callout-caution}
### 📖 题目描述
给定一个字符串 `s` ，请你找出其中不含有重复字符的 **最长子串** 的长度。

**输入输出示例**：

* **示例 1**：
    * **输入**：`s = "abcabcbb"`
    * **输出**：`3`
    * **解释**：因为无重复字符的最长子串是 `"abc"`，所以其长度为 3。

* **示例 2**：
    * **输入**：`s = "bbbbb"`
    * **输出**：`1`
    * **解释**：因为无重复字符的最长子串是 `"b"`，所以其长度为 1。

* **示例 3**：
    * **输入**：`s = "pwwkew"`
    * **输出**：`3`
    * **解释**：因为无重复字符的最长子串是 `"wke"`，所以其长度为 3。请注意，你的答案必须是 子串 的长度，`"pwke"` 是一个子序列，不是子串。
:::

In [9]:
class Solution3:
    def lengthOfLongestSubstring(self, s: str) -> int:
        char_index_map = {}  # 记忆中枢：记录字符最近一次出现的位置
        left = 0             # 窗口左边界
        max_length = 0       # 历史最长记录
        for right, char in enumerate(s):
            # 查档：如果在哈希表中找到，且它以前的位置在当前窗口内
            if char in char_index_map and char_index_map[char] >= left:
                left = char_index_map[char] + 1  # 左指针发动空间跃迁，直接瞬移到重复字符的下一位！
            char_index_map[char] = right  # 将当前字符的最新位置记入哈希表
            max_length = max(max_length, right - left + 1)  # 计算当前窗口长度，并与历史记录打擂台
        return max_length

In [12]:
#| code-fold: true
def test_3(func):
    test_cases = [
        {"desc": "1. 示例 1: 常规穿插", "s": "abcabcbb", "expected": 3},
        {"desc": "2. 示例 2: 全重复单字符", "s": "bbbbb", "expected": 1},
        {"desc": "3. 示例 3: 中间出现重复", "s": "pwwkew", "expected": 3},
        {"desc": "4. 边界: 空字符串", "s": "", "expected": 0},
        {"desc": "5. 边界: 只有空格", "s": " ", "expected": 1},
        {"desc": "6. 边界: 全不相同", "s": "abcdefg", "expected": 7},
        {"desc": "7. 瞬移坑点: (abba) 测试左指针不倒退", "s": "abba", "expected": 2},
        {"desc": "8. 数字与符号混合", "s": "a1!ba1!", "expected": 4},
        {"desc": "9. 长字符串压测", "s": "a" * 1000 + "bcde" + "f" * 1000, "expected": 6}
    ]

    passed = 0
    print(f"{'ID':<3} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 70)

    for i, tc in enumerate(test_cases):
        actual = func(tc["s"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<3} | {status:<6} | {str(tc['expected']):<4} | {str(actual):<4} | {tc['desc']}")

    print("-" * 70)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_3(Solution3().lengthOfLongestSubstring)

ID  | 结果     | 预期   | 实际   | 描述
----------------------------------------------------------------------
1   | ✅ PASS | 3    | 3    | 1. 示例 1: 常规穿插
2   | ✅ PASS | 1    | 1    | 2. 示例 2: 全重复单字符
3   | ✅ PASS | 3    | 3    | 3. 示例 3: 中间出现重复
4   | ✅ PASS | 0    | 0    | 4. 边界: 空字符串
5   | ✅ PASS | 1    | 1    | 5. 边界: 只有空格
6   | ✅ PASS | 7    | 7    | 6. 边界: 全不相同
7   | ✅ PASS | 2    | 2    | 7. 瞬移坑点: (abba) 测试左指针不倒退
8   | ✅ PASS | 4    | 4    | 8. 数字与符号混合
9   | ✅ PASS | 6    | 6    | 9. 长字符串压测
----------------------------------------------------------------------
测试总结: 通过 9/9


::: {.callout-important}
### 💡 思路讲解

1. **确立双指针窗口**：
   - 使用左指针 `left` 和右指针 `right` 维护一个滑动窗口 `[left, right]`。这个窗口内的规则绝对严格：**决不允许有任何重复字符**。
   - `right` 指针负责在前面开疆拓土，每前进一步，就考察一个新字符。

2. **哈希表化身“记忆中枢”**：
   - 创建哈希表 `char_index_map`，专门用来记录每个字符 **最后一次出现的位置索引**（即 `{"字符": 索引}`）。

3. **极速瞬移逻辑（核心降维点）**：
   - 当 `right` 遇到一个字符 `char` 时，先去哈希表里查档：这个字符以前来过吗？
   - **如果没来过，或者来过但它以前的位置在当前左指针的左边（即不在当前窗口内了，已经过期了）**：一切安全，把当前字符的最新位置登记到哈希表，并计算窗口长度。
   - **⚠️ 如果来过，且它的旧位置 `>= left`（说明当前窗口内出现了内鬼）**：
     - 传统做法：让 `left` 一步一步往右挪，直到把那个重复的内鬼踢出去。
     - **哈希表降维做法**：既然哈希表知道那个内鬼的具体位置 `旧索引`，我们直接让 `left` **瞬间移动**到 `旧索引 + 1` 的位置！一步到位，极其霸道！
   - 最后，无论是否冲突，都用 `right` 的当前索引覆盖哈希表中的旧索引，并更新最长记录 `max_length`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(n)$。其中 $n$ 为字符串的长度。得益于哈希表的瞬移指挥，`right` 指针只遍历数组一次，`left` 指针发生冲突时直接跳跃，绝不回头。整个过程时间复杂度被压缩到了极致的线性时间。
* **空间复杂度**：$O(\Sigma)$。其中 $\Sigma$ 表示字符集（即字符串中出现的独立字符种类数）。如果是英文字母、数字和符号，哈希表最多存储 128 个（ASCII码数量）键值对，因此空间复杂度实际上是极小的 $O(1)$ 常数级别。
:::